# SubGen on Colab

Sentence-level `.srt` subtitles from English speech (WhisperX), plus a 1080p subtitle video
for YouTube with your own clips as the background.

1. **Runtime → Change runtime type → L4 GPU** (any NVIDIA GPU works; L4 is fastest here).
2. Run the setup cells, then either the **Drive pipeline** or the **web UI**.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
![ -d SubGen ] || git clone https://github.com/enoky/SubGen.git
%cd /content/SubGen
!git pull --ff-only

In [ ]:
%%bash
# Install into Colab's own Python, keeping its preinstalled CUDA build of torch when
# WhisperX accepts it (the repo's lockfile targets Windows CUDA 13.2 wheels).
pip install -q uv
python -c "import torch, torchaudio; print(f'torch=={torch.__version__}'); print(f'torchaudio=={torchaudio.__version__}')" > /tmp/torch-pins.txt
PKGS=("whisperx>=3.7.2" "deepmultilingualpunctuation>=1.0.1" "spacy>=3.8,<3.9" "srt>=3.5.3" "tqdm" "gradio" "pillow"
      "en-core-web-lg @ https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.8.0/en_core_web_lg-3.8.0-py3-none-any.whl")
uv pip install --system -q -c /tmp/torch-pins.txt "${PKGS[@]}" \
  || { echo "Colab's torch is too old for WhisperX; letting it upgrade torch."; uv pip install --system -q "${PKGS[@]}"; }

### GPU video check

Video decoding (NVDEC), scaling and subtitle overlay (CUDA) and encoding (NVENC) run on the
GPU. Each GPU route is checked on one second of video before the full encode and skipped if it
leaves edge artifacts. If Colab's ffmpeg lacks NVENC, this installs a static ffmpeg build that
has it. Either way, the pipeline falls back to slower paths automatically if something isn't
available.

In [ ]:
%%bash
cd /content/SubGen
if python -c "import srt2mp4, sys; c = srt2mp4.ffmpeg_caps(); sys.exit(0 if 'hevc_nvenc' in c.encoders else 1)"; then
  echo "Colab's ffmpeg has NVENC."
else
  echo "Installing a static ffmpeg build with NVENC/NVDEC..."
  URL=https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-n7.1-latest-linux64-gpl-7.1.tar.xz
  if curl -fsSL -o /tmp/ffmpeg.tar.xz "$URL"; then
    mkdir -p /opt/ffmpeg && tar -xJf /tmp/ffmpeg.tar.xz -C /opt/ffmpeg --strip-components=1
    ln -sf /opt/ffmpeg/bin/ffmpeg /usr/local/bin/ffmpeg
    ln -sf /opt/ffmpeg/bin/ffprobe /usr/local/bin/ffprobe
  else
    echo "Download failed; keeping Colab's ffmpeg (slower fallback paths will be used)."
  fi
fi
ffmpeg -version | head -1
python -m srt2mp4 --check

### Drive pipeline (no UI)

Put your files in Google Drive like this:

```
MyDrive/subgen/
  narration.wav        narration audio
  clips/*.mp4          background clips, joined in filename order (clip2 before clip10)
  script.txt           optional: original script, used for the exact cue wording
  528Hz_ambient.m4a    optional: background music, mixed 20 LU under the narration
  output/              narration.srt, narration.mp4 and narration_norm.wav are saved here
```

The narration is first normalized to -14 LUFS (YouTube's loudness reference); the
normalized audio is transcribed, used in the video and saved as `output/narration_norm.wav`.
The clips are center-cropped to 16:9 and scaled to 1920x1080; their audio is dropped. If they
run shorter than the narration, the last frame is held; if longer, they are cut. The video is
HEVC (NVENC), sized for YouTube 1080p uploads.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content/SubGen
# Extra options are passed to subgen, e.g. --model large-v3-turbo.
# --reuse-srt re-renders the video from output/narration.srt (e.g. after editing it on Drive).
# Clips are copied locally first (16 at a time); --stream-clips reads them from Drive during encoding instead.
# --benchmark times each video route and the Drive read speed on ~40 s of clips, then stops.
# --check-routes only checks which video route works here (~1 min), then stops.
# --music-level -24 makes the background music quieter (default -20 LU under the narration); --no-music drops it.
# --tint-opacity 0.3 lightens the indigo wash over the clips (default 0.45; 0 = off); --tint "#RRGGBB" changes its colour.
# Subtitle times are snapped to the speech (also for a reused SRT; original kept as narration.srt.bak); --no-retime keeps them as is.
# --srt-only skips the video. --no-normalize keeps narration.wav as is; --lufs -16 changes the target.
!python -m drive_pipeline --root /content/drive/MyDrive/subgen

### Optional: YouTube Short

Same pipeline for a 1080x1920 Short. Put this Short's files in their own Drive folder:

```
MyDrive/subgen_shorts/
  narration.wav        narration audio (up to 3 minutes for a Short)
  clips/*.mp4          portrait clips: 1088x1920 or 704x1280 (LTX)
  script.txt           optional
  output/              narration.srt, narration_phrases.srt and narration.mp4 are saved here
```

Portrait clips switch to the Shorts layout automatically. The background music from
`MyDrive/subgen/` is used unless this folder has its own.

The Short shows a few words at a time (short phrases timed to the speech); add `--sentences` for whole sentences. Phrases sit at the bottom, just above the Shorts title area; the footage is untinted; the background music starts 10 s into the track with 2 s fades in and out.

In [ ]:
%cd /content/SubGen
# Same options as above, e.g. --reuse-srt to re-render after editing output/narration.srt.
!python -m drive_pipeline --root /content/drive/MyDrive/subgen_shorts --format shorts

### Optional: web UI

Prints a public `gradio.live` link. To cache models on Drive, put
`/content/drive/MyDrive/subgen-models` in the app's **Model cache directory** setting
(mount Drive above first).

In [ ]:
%cd /content/SubGen
from subgen_gradio import build_app

build_app().queue().launch(share=True, debug=True)

### Optional: small 720p AV1 copy

Re-encodes `output/narration.mp4` to a 720p AV1 file at about 500 kbps
(`output/narration_720p_av1_500k.mp4`) on the GPU (`av1_nvenc`, supported on L4).
It's written to local disk first (so `+faststart`'s second pass doesn't go through Drive),
then copied to Drive. Progress is printed every 10 seconds. Run the Drive pipeline first.

In [ ]:
!ffmpeg -stats_period 10 -progress - -nostats -i "/content/drive/MyDrive/subgen/output/narration.mp4" \
  -vf "scale=-2:720:flags=lanczos" \
  -c:v av1_nvenc -preset p7 -tune hq -rc vbr -multipass fullres \
  -b:v 500k -maxrate 750k -bufsize 1500k \
  -spatial-aq 1 -temporal-aq 1 -rc-lookahead 32 -bf 4 -b_ref_mode middle -g 240 \
  -pix_fmt yuv420p \
  -c:a aac -b:a 96k -ac 2 \
  -movflags +faststart -y "/content/narration_720p_av1_500k.mp4" \
  && cp "/content/narration_720p_av1_500k.mp4" "/content/drive/MyDrive/subgen/output/" \
  && echo "Saved /content/drive/MyDrive/subgen/output/narration_720p_av1_500k.mp4"

### Optional: 1080p AV1 copy

Re-encodes `output/narration.mp4` to a 1080p AV1 file at about 1000 kbps
(`output/narration_1080p_av1_1000k.mp4`) on the GPU (`av1_nvenc`). It's written to local
disk first, then copied to Drive. Progress is printed every 10 seconds. Run the Drive
pipeline first.

In [ ]:
!ffmpeg -stats_period 10 -progress - -nostats -i "/content/drive/MyDrive/subgen/output/narration.mp4" \
  -c:v av1_nvenc -preset p7 -tune hq -rc vbr -multipass fullres \
  -b:v 1000k -maxrate 1500k -bufsize 2500k \
  -spatial-aq 1 -temporal-aq 1 -rc-lookahead 32 -bf 4 -b_ref_mode middle -g 240 \
  -pix_fmt yuv420p \
  -c:a aac -b:a 128k -ac 2 \
  -movflags +faststart -y "/content/narration_1080p_av1_1000k.mp4" \
  && cp "/content/narration_1080p_av1_1000k.mp4" "/content/drive/MyDrive/subgen/output/" \
  && echo "Saved /content/drive/MyDrive/subgen/output/narration_1080p_av1_1000k.mp4"